# Construir Aplicações de Geração de Imagens (Azure OpenAI)

Os LLMs não servem só para gerar texto. Também pode gerar imagens a partir de descrições de texto. Imagens, enquanto modalidade, são úteis em diversas áreas como MedTech, arquitetura, turismo, desenvolvimento de jogos, marketing e mais. Nesta lição, trabalhamos com os atuais modelos **GPT Image** na Microsoft Foundry.

## Objetivos de aprendizagem

No final desta lição, será capaz de:

- Explicar o que é geração de imagens e onde é útil.
- Compreender a família de modelos `gpt-image` e como ela difere dos modelos legados DALL·E.
- Construir uma aplicação de geração de imagens e editar imagens.

## O que é geração de imagens?

Modelos de geração de imagens criam imagens a partir de um prompt de texto. Modelos modernos como o `gpt-image` aprendem, durante o treino, a relação entre texto e imagens, e depois convertem iterativamente ruído aleatório numa imagem que corresponde ao seu prompt.

Duas famílias bem conhecidas de modelos de imagens são:

- **`gpt-image` (OpenAI)** — a geração atual usada nesta lição. Suporta geração de imagem a partir de texto e edição de imagens (inpainting com uma máscara).
- **Midjourney** — um modelo popular de terceiros com serviço próprio e fluxo de trabalho baseado no Discord.

> Os modelos OpenAI mais antigos — **DALL·E 2** e **DALL·E 3** — são legados. O DALL·E 3 já não está disponível para novas implementações, e funcionalidades como `create_variation` existiam só no DALL·E 2. Use os modelos `gpt-image` para novas aplicações.

Na Microsoft Foundry, o **`gpt-image-2`** é o modelo de imagem mais recente e capaz, sendo o predefinido recomendado. Os `gpt-image-1.5` e `gpt-image-1-mini` também estão geralmente disponíveis.

> **Importante:** os modelos `gpt-image` devolvem a imagem gerada em **base64** (`b64_json`), não como URL. O seu código decodifica a string base64 para bytes e guarda-a — não há URL de imagem para descarregar.


## Construir a sua primeira aplicação de geração de imagens

Então, o que é necessário para construir uma aplicação de geração de imagens? Você precisa das seguintes bibliotecas:

- **python-dotenv**, é altamente recomendável usar esta biblioteca para manter os seus segredos num ficheiro *.env* separado do código.
- **openai**, esta biblioteca é o que vai utilizar para interagir com a API da OpenAI.
- **pillow**, para trabalhar com imagens em Python.

Se ainda não o fez, siga as instruções na página [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) para criar um recurso e modelo Microsoft Foundry. Selecione **gpt-image-2** como modelo (o mais recente modelo Azure OpenAI para imagens; DALL·E é legado).

1. Crie um ficheiro *.env* com o seguinte conteúdo:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Localize esta informação no portal Microsoft Foundry para o seu recurso na secção "Deployments".



1. Reúna as bibliotecas acima num ficheiro chamado *requirements.txt* da seguinte forma:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Em seguida, crie um ambiente virtual e instale as bibliotecas:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Para Windows, use os seguintes comandos para criar e ativar o seu ambiente virtual:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Adicione o seguinte código no ficheiro chamado *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # configurar o cliente Azure OpenAI (Microsoft Foundry).
    # Os modelos de imagem necessitam de uma versão recente da API — verifique os documentos do Foundry para saber qual a versão que o seu modelo requer.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Criar uma imagem usando a API de geração de imagens
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Insira o texto do seu prompt aqui
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # por exemplo, "gpt-image-2"
        )
        # Definir o diretório para armazenar a imagem
        image_dir = os.path.join(os.curdir, 'images')

        # Se o diretório não existir, criá-lo
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializar o caminho da imagem (note que o tipo de ficheiro deve ser png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Os modelos gpt-image retornam a imagem em base64 (b64_json), não numa URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Exibir a imagem no visualizador de imagens padrão
        image = Image.open(image_path)
        image.show()

    # capturar exceções
    except BadRequestError as err:
        print(err)

    ```

Vamos explicar este código:

- Primeiro, importamos as bibliotecas de que precisamos, incluindo a biblioteca OpenAI, a biblioteca dotenv, o módulo base64, e a biblioteca Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- De seguida, carregamos as variáveis de ambiente a partir do ficheiro *.env*.

    ```python
    # importar dotenv
    dotenv.load_dotenv()
    ```

- Depois, configuramos o cliente Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Em seguida, geramos a imagem:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Introduza o texto do seu prompt aqui
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Os modelos `gpt-image` devolvem a imagem como uma string **base64** em `data[0].b64_json`. Decodificamo-la para bytes e escrevemo-la num ficheiro — não existe um URL para descarregar.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Por fim, abrimos a imagem e usamos o visualizador de imagens padrão para a mostrar:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Mais detalhes sobre a geração da imagem

Vamos ver os parâmetros de `images.generate`:

- **prompt**, é o prompt de texto usado para gerar a imagem. Aqui é "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, é o tamanho da imagem gerada (por exemplo `1024x1024`, `1536x1024`, `1024x1536`, ou `"auto"`).
- **n**, é o número de imagens geradas. Aqui geramos uma.
- **model**, é o nome do seu deployment do modelo de imagem (por exemplo `gpt-image-2`).

> Os modelos de imagem não têm um parâmetro `temperature` — isso é um controlo de geração de texto. Para obter variedade, chame a API novamente; para reduzir variedade, torne o seu prompt mais específico.

## Capacidades adicionais da geração de imagens

Viu como gerar uma imagem com poucas linhas de Python. Os modelos `gpt-image` podem também **editar** uma imagem existente. Ao fornecer uma imagem, uma **máscara** opcional (que marca a área a alterar), e um prompt, pode alterar parte de uma imagem — por exemplo, adicionar um chapéu ao nosso coelho.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# as edições também são devolvidas como base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

A imagem base contém apenas o coelho; a imagem final tem o chapéu no coelho.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
